<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/07_hat_x3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HAT ×3 — a larger Transformer

A blurred satellite image at 10 m, made three times finer — 3.33 m.

## 1. Data

In [ ]:
import sys, urllib.request
!pip install -q timm einops

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'hat_arch.py', 'hat_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # drop a stale copy if one is already imported

from sr_utils import *

show_data()        # 2 validation patches + 2 test areas

## 2. Training

**This is only here to show that the code runs.**  16 images, one epoch.
The results below come from weights trained on the full dataset.

Heavier than SwinIR: a full 128 px input takes 12 GB, which a Colab T4 can barely
hold.  Here it runs on 64 px crops at batch 1, about 3 GB.  The real training used
48 px crops.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from hat_models import build_hat

# the crop must be a multiple of 16 — HAT reads the input in 16x16 windows
N_TRAIN, EPOCHS, BATCH, CROP = 16, 1, 1, 64
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
x, y = to_t(lo)[:, :, :CROP, :CROP], to_t(hi)[:, :, :CROP * 3, :CROP * 3]
loader = DataLoader(TensorDataset(x, y), batch_size=BATCH, shuffle=True)

net = build_hat(3).to(dev).train()
opt = torch.optim.Adam(net.parameters(), 2e-4, betas=(0.9, 0.99))
crit = nn.L1Loss()

for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for xb, yb in loader:
        loss = crit(net(xb.to(dev)), yb.to(dev))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   L1 {tot/len(loader):.5f}')

del net, opt, loader, x, y
torch.cuda.empty_cache()      # make room for the trained weights in the next cell

## 3. Results

In [ ]:
MODEL = f'{BASE}/models/07_hat_x3'

from hat_models import load_hat, hat_upscale

net = load_hat(fetch(f'{MODEL}/checkpoints/hat_x3.pth', 'hat_x3.pth'))
print(f'HAT  {sum(p.numel() for p in net.parameters())/1e6:.2f}M')

upscale = lambda lr: hat_upscale(net, lr)

show_results(upscale, 'HAT', center=[(67, 370), (313, 36)])

## 4. Final test

A real Sentinel-2 capture, with no ground truth.  Judged by eye.


In [ ]:
show_test(upscale, 'HAT', center=(800, 800), size=70)